# **Coleta de Sumulas da CBF**

## **1. Configuracao do Ambiente**

In [1]:
import re
import time
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import requests
import urllib3
from urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(InsecureRequestWarning)

## **2. Configuracao**

In [2]:
BASE_URL = "https://www.cbf.com.br"
API_URL = f"{BASE_URL}/api/cbf/jogos/campeonato"
DATA_PATH = Path("../data")
RAW_DATA_PATH = DATA_PATH / "raw"
MATCH_REPORTS_PATH = RAW_DATA_PATH / "match-reports"
VIZ_PATH = Path("../viz")

YEARS = list(range(2020, 2027))
COMPETITION_IDS = {
    2020: "12464",
    2021: "12487",
    2022: "12518",
    2023: "12555",
    2024: "12584",
    2025: "12606",
    2026: "1260611",
}
MAX_ROUNDS = 38
REQUEST_DELAY = 0.2

for path in [MATCH_REPORTS_PATH, VIZ_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Diretorios criados:")
print(f"  Match reports: {MATCH_REPORTS_PATH}")
print(f"  Viz: {VIZ_PATH}")

Diretorios criados:
  Match reports: ../data/raw/match-reports
  Viz: ../viz


## **3. Funcoes Utilitarias**

In [3]:
def normalize_text(value: str) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(char for char in text if not unicodedata.combining(char))
    return text.casefold().strip()


def slugify(value: str) -> str:
    text = normalize_text(value)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-") or "sem-nome"


def safe_int(value, default: int = 0) -> int:
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return default


def fetch_json(session: requests.Session, url: str, max_retries: int = 3) -> Optional[dict]:
    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            if attempt == max_retries - 1:
                print(f"Falha em {url}: {error}")
                return None
            time.sleep((attempt + 1) * 2)
    return None

## **4. Coleta de Documentos**

In [4]:
def flatten_documents(payload: dict, year: int, competition_id: str, round_number: int) -> List[Dict[str, str]]:
    rows = []
    for group in payload.get("jogos", []) or []:
        for game in group.get("jogo", []) or []:
            base = {
                "year": year,
                "competition_id": competition_id,
                "round": safe_int(game.get("rodada"), round_number),
                "game_id": game.get("id_jogo"),
                "game_number": safe_int(game.get("num_jogo")),
                "home_team": game.get("mandante", {}).get("nome"),
                "away_team": game.get("visitante", {}).get("nome"),
                "match_date": str(game.get("data", "")).strip(),
                "match_time": str(game.get("hora", "")).strip(),
                "venue": game.get("local"),
            }
            for document in game.get("documentos", []) or []:
                title = document.get("title", "")
                rows.append({
                    **base,
                    "document_title": title,
                    "document_title_normalized": normalize_text(title),
                    "document_url": document.get("url", ""),
                })
    return rows


def collect_documents() -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False
    rows = []
    for year in YEARS:
        competition_id = COMPETITION_IDS[year]
        for round_number in range(1, MAX_ROUNDS + 1):
            url = f"{API_URL}/{competition_id}/rodada/{round_number}/fase"
            payload = fetch_json(session, url)
            if payload:
                rows.extend(flatten_documents(payload, year, competition_id, round_number))
            time.sleep(REQUEST_DELAY)
        print(f"Ano {year}: rodadas processadas")
    data = pd.DataFrame(rows)
    if not data.empty:
        data = data.sort_values(["year", "round", "game_number", "document_title"]).reset_index(drop=True)
    return data

In [5]:
documents = collect_documents()
documents_index_path = MATCH_REPORTS_PATH / "cbf_documents_2020_2026.csv"
documents.to_csv(documents_index_path, index=False)

print(f"Documentos encontrados: {len(documents)}")
print(f"Indice salvo em: {documents_index_path}")
documents.head()

Ano 2020: rodadas processadas
Ano 2021: rodadas processadas
Ano 2022: rodadas processadas
Ano 2023: rodadas processadas
Ano 2024: rodadas processadas
Ano 2025: rodadas processadas
Ano 2026: rodadas processadas
Documentos encontrados: 7281
Indice salvo em: ../data/raw/match-reports/cbf_documents_2020_2026.csv


,year,competition_id,round,game_id,game_number,home_team,away_team,match_date,match_time,venue,document_title,document_title_normalized,document_url
0,2020,12464,1,815072,1,Flamengo,Atlético Mineiro,09/08/2020,16:00,Maracanã - Rio de Janeiro - RJ,Boletim Financeiro,boletim financeiro,https://conteudo.cbf.com.br/sumulas/2020/1421b...
1,2020,12464,1,815072,1,Flamengo,Atlético Mineiro,09/08/2020,16:00,Maracanã - Rio de Janeiro - RJ,Relatório de Jogo,relatorio de jogo,https://conteudo.cbf.com.br/sumulas/2020/1421r...
2,2020,12464,1,815072,1,Flamengo,Atlético Mineiro,09/08/2020,16:00,Maracanã - Rio de Janeiro - RJ,Súmula,sumula,https://conteudo.cbf.com.br/sumulas/2020/1421s...
3,2020,12464,1,815073,2,Botafogo de Futebol E Regatas,Esporte Clube Bahia,30/09/2020,21:30,Nilton Santos - Rio de Janeiro - RJ,Boletim Financeiro,boletim financeiro,https://conteudo.cbf.com.br/sumulas/2020/1422b...
4,2020,12464,1,815073,2,Botafogo de Futebol E Regatas,Esporte Clube Bahia,30/09/2020,21:30,Nilton Santos - Rio de Janeiro - RJ,Relatório de Jogo,relatorio de jogo,https://conteudo.cbf.com.br/sumulas/2020/1422r...


## **5. Download das Sumulas**

In [6]:
def build_match_report_path(row: pd.Series) -> Path:
    home = slugify(row["home_team"])
    away = slugify(row["away_team"])
    year = int(row["year"])
    round_number = int(row["round"])
    game_number = int(row["game_number"])
    game_id = row["game_id"]
    filename = f"{year}_r{round_number:02d}_j{game_number:03d}_{home}_x_{away}_{game_id}_sumula.pdf"
    return MATCH_REPORTS_PATH / str(year) / f"round-{round_number:02d}" / filename


def download_pdf(session: requests.Session, url: str, path: Path, max_retries: int = 3) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return "exists"
    temp_path = path.with_suffix(path.suffix + ".tmp")
    for attempt in range(max_retries):
        try:
            with session.get(url, timeout=60, stream=True) as response:
                response.raise_for_status()
                with temp_path.open("wb") as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            file.write(chunk)
            temp_path.replace(path)
            return "downloaded"
        except requests.RequestException as error:
            if temp_path.exists():
                temp_path.unlink()
            if attempt == max_retries - 1:
                print(f"Falha no download {url}: {error}")
                return "failed"
            time.sleep((attempt + 1) * 2)
    return "failed"


def collect_match_reports(documents_df: pd.DataFrame) -> pd.DataFrame:
    reports = documents_df[documents_df["document_title_normalized"].eq("sumula")].copy()
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False
    reports["file_path"] = ""
    reports["download_status"] = ""
    for position, (index, row) in enumerate(reports.iterrows(), start=1):
        file_path = build_match_report_path(row)
        status = download_pdf(session, row["document_url"], file_path)
        reports.loc[index, "file_path"] = str(file_path)
        reports.loc[index, "download_status"] = status
        if position % 50 == 0:
            print(f"Progresso: {position}/{len(reports)} sumulas")
        time.sleep(REQUEST_DELAY)
    return reports.reset_index(drop=True)

In [7]:
match_reports = collect_match_reports(documents)
match_reports_index_path = MATCH_REPORTS_PATH / "match_reports_index.csv"
match_reports.to_csv(match_reports_index_path, index=False)

print(f"Sumulas encontradas: {len(match_reports)}")
print(f"Indice salvo em: {match_reports_index_path}")
match_reports["download_status"].value_counts(dropna=False)

Progresso: 50/2427 sumulas
Progresso: 100/2427 sumulas
Progresso: 150/2427 sumulas
Progresso: 200/2427 sumulas
Progresso: 250/2427 sumulas
Progresso: 300/2427 sumulas
Progresso: 350/2427 sumulas
Progresso: 400/2427 sumulas
Progresso: 450/2427 sumulas
Progresso: 500/2427 sumulas
Progresso: 550/2427 sumulas
Progresso: 600/2427 sumulas
Progresso: 650/2427 sumulas
Progresso: 700/2427 sumulas
Progresso: 750/2427 sumulas
Progresso: 800/2427 sumulas
Progresso: 850/2427 sumulas
Progresso: 900/2427 sumulas
Progresso: 950/2427 sumulas
Progresso: 1000/2427 sumulas
Progresso: 1050/2427 sumulas
Progresso: 1100/2427 sumulas
Progresso: 1150/2427 sumulas
Progresso: 1200/2427 sumulas
Progresso: 1250/2427 sumulas
Progresso: 1300/2427 sumulas
Progresso: 1350/2427 sumulas
Progresso: 1400/2427 sumulas
Progresso: 1450/2427 sumulas
Progresso: 1500/2427 sumulas
Progresso: 1550/2427 sumulas
Progresso: 1600/2427 sumulas
Progresso: 1650/2427 sumulas
Progresso: 1700/2427 sumulas
Progresso: 1750/2427 sumulas
Progr

download_status
exists    2427
Name: count, dtype: int64